# Week 2, Day 1: OpenAI Agents SDK

## What this lab covers

This lab configures the OpenAI Agents SDK with an Azure OpenAI-compatible model, runs a first agent, and records execution with traces. It then adds a custom tool and compares two ways to preserve conversation memory: manually carrying message history and using a SQLite-backed session.

## Part 01: Creating an Agent using OpenAI SDK

### OpenAI Agents SDK Uses

In [10]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession, OpenAIChatCompletionsModel, set_tracing_disabled
from openai import AsyncAzureOpenAI 
from IPython.display import Markdown, display
import asyncio
import datetime

set_tracing_disabled(True)  # Disable tracing for this notebook to avoid cluttering the output
load_dotenv(override=True) # Load environment variables from .env file, override existing ones if necessary

True

In [2]:
# Let's see if the API key is working/helping us to call LLM from Azure Foundry
import os
# From OpenAI
AZURE_OPENAI_API_KEY= os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_MODEL_DEPLOYMENT = os.getenv("AZURE_OPENAI_DEPLOYMENT")
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
AZURE_OPENAI_DEPLOYMENT_GPT_41 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_41")
AZURE_OPENAI_DEPLOYMENT_GPT_54_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_54_mini")
AZURE_OPENAI_DEPLOYMENT_GPT_55 = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_55")
AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini = os.getenv("AZURE_OPENAI_DEPLOYMENT_GPT_4O_mini")
if AZURE_OPENAI_API_KEY:
    print("AZURE_OPENAI_API_KEY is available")
else:
    print("AZURE_OPENAI_API_KEY is not available")

# From Anthropic
AZURE_CLAUDE_DEPLOYMENT_OPUS_48=os.getenv("AZURE_CLAUDE_DEPLOYMENT_OPUS_48")
AZURE_CLAUDE_ENDPOINT=os.getenv("AZURE_CLAUDE_ENDPOINT")
AZURE_CLAUDE_API_KEY=os.getenv("AZURE_CLAUDE_API_KEY")
if AZURE_CLAUDE_API_KEY:
    print("AZURE_CLAUDE_API_KEY is avaiable")
else:
    print("AZURE_CLAUDE_API_KEY is not available")



AZURE_OPENAI_API_KEY is available
AZURE_CLAUDE_API_KEY is avaiable


In [3]:
## setting up the client for OpenAI using Azure Endpoint (Azure Foundry)
client_openai = AsyncAzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION
)
## let's a define a model pointing to Azure deployment
model_openai = OpenAIChatCompletionsModel(
    openai_client=client_openai,
    model=AZURE_OPENAI_DEPLOYMENT_GPT_41
)

In [4]:
## Let's make an agent with name, instruction, model
## Here 'instructions' param is like passing a system prompt to model
agent = Agent(name="Highly Educated Person", instructions="You are a highly educated person", model=model_openai)

In [ ]:
# let's call this agent
result = await Runner.run(agent, "Tell me a joke about PC games")
print(result)

In [14]:
display(Markdown(result.final_output))

Why did the gamer take a ladder to the computer?

Because they heard the graphics were on another level!

In [ ]:
## Here is detail of the LLM calls
result.to_input_list()

[{'content': 'Tell me a joke about PC games', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Why did the gamer take a ladder to the computer?\n\nBecause they heard the graphics were on another level!',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gpt-4.1',
   'response_id': 'chatcmpl-E3IZInEG7SazLZaKgvl79c6gECOrT'}}]

### Adding Observability with a trace

In [ ]:
## Here we need to have set up a account in OpenAI to get trace of agent calls.
## But in my case we could either make custom tracer, use Azure Foundry for tracing, or use other tracer like ML Flow etc.
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell me a joke around programmers")
print(result.final_output)

Why do programmers prefer dark mode?

Because light attracts bugs!


OPENAI_API_KEY is not set, skipping trace export


In [5]:
## Let's see about streaming
result = Runner.run_streamed(agent, "Tell me 3 things about humans which you think could be improved")
result_details = ""
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end = "", flush = True)



Certainly! Here are three aspects of humans that could potentially be improved, considering both individual and societal perspectives:

1. **Emotional Regulation and Empathy**  
Humans sometimes struggle to manage emotions like anger, anxiety, or jealousy, which can lead to interpersonal conflict or poor decision-making. Enhanced emotional regulation and empathy would foster better relationships, reduce misunderstandings, and promote cooperation both

OPENAI_API_KEY is not set, skipping trace export


 personally and globally.

2. **Critical Thinking and Rationality**  
Humans are prone to cognitive biases, logical fallacies, and sometimes misinformation. Improving critical thinking skills—questioning assumptions, evaluating evidence, and reasoning logically—would lead to better decisions, more productive debates, and progress in science and society.

3. **Sustainability and Long-term Planning**  
As a species, humans often prioritize immediate needs and desires over the long-term well-being of themselves, their communities, and the planet. Enhancing the ability to plan for the future, consider environmental impacts, and act sustainably would help address challenges like climate change, resource depletion, and global inequality.

These improvements could not only benefit individuals but also create a more harmonious, rational, and sustainable society.

OPENAI_API_KEY is not set, skipping trace export


## Part 02: Adding a tool

In [9]:
# let's create some custom tools with Python using OpenAI SDK 
@function_tool # help's making json format, required for AI model to understand about the function/tool
def record_message_tool(topic: str, message: str) -> str:
    """ Record the user message with it's topic in a file  """
    print(f"Tool called to record an message: '{message}' on topic '{topic}'")
    # setting up the timestamp and entry format
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    entry = f"\n\n ## [{timestamp}] - TOPIC: {topic}\n* **Content:** {message}"

    with open("ai_model_message.txt","a", encoding="utf-8") as file:
        file.write(entry)
    return f"Successfully saved message on topic '{topic}'."

In [5]:
record_message_tool

FunctionTool(name='record_message_tool', description="Record the user message with it's topic in a file", params_json_schema={'properties': {'topic': {'title': 'Topic', 'type': 'string'}, 'message': {'title': 'Message', 'type': 'string'}}, 'required': ['topic', 'message'], 'title': 'record_message_tool_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x000001F2C32DD540>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None)

In [6]:
# let's look at the tool's schema
record_message_tool.params_json_schema

{'properties': {'topic': {'title': 'Topic', 'type': 'string'},
  'message': {'title': 'Message', 'type': 'string'}},
 'required': ['topic', 'message'],
 'title': 'record_message_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [10]:
# Let's have a look at the description
record_message_tool.description

"Record the user message with it's topic in a file"

In [7]:
# creating a new agent
notifier_agent = Agent(
    name = "Notifier", 
    model = model_openai, 
    instructions="You will record the user message",
    tools = [record_message_tool]
)

In [8]:
# Try an use this newly created agent with it's custom tool
result = await Runner.run(
    starting_agent = notifier_agent,
    input = "I have a brought the pizza, could you pass on this information to respective person"
)
print(result.final_output)

OPENAI_API_KEY is not set, skipping trace export


Tool called to record an message: The pizza has arrived. on topic Food delivery update
The information that the pizza has arrived has been recorded and passed on to the respective person. If you need me to notify anyone specifically or add extra details, let me know!


OPENAI_API_KEY is not set, skipping trace export


In [11]:
# Try an use this newly created agent with it's custom tool
result = await Runner.run(
    starting_agent = notifier_agent,
    input = "Hey, I am Kamal, your neighbour. I just came to visit you, you there..?"
)
print(f"Agent results: {result.final_output}")

Tool called to record an message: Hey, I am Kamal, your neighbour. I just came to visit you, you there..? on topic Neighbour Visit
Agent results: Hi Kamal! Yes, I'm here. Thanks for dropping by—it's great to see you. What brings you over today?


## Part 03: Sessions (memory)

#### Memory Approach 01: Just manually pass in the list of dicts

In [12]:
 # define a simple agent 
agent = Agent(
    name = "Assitant",
    model = model_openai
)
# requesting response from AI 
response = await Runner.run(
    starting_agent = agent,
    input = "Hi there, my name is Hardeep!"
)
print(response.final_output)

Hello, Hardeep! It’s great to meet you. How can I assist you today?


In [15]:
response.to_input_list()

[{'content': 'Hi there, my name is Hardeep!', 'role': 'user'},
 {'id': '__fake_id__',
  'content': [{'annotations': [],
    'text': 'Hello, Hardeep! It’s great to meet you. How can I assist you today?',
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message',
  'provider_data': {'model': 'gpt-4.1',
   'response_id': 'chatcmpl-E3MnzGnpDJYjFQrQfrWkNOZjchQFM'}}]

In [16]:
next_input = response.to_input_list() + [
    {
        "role": "user",
        "content": "What's my name?"
    }
]
# requesting response from AI 
response = await Runner.run(
    starting_agent = agent,
    input = next_input
)
print(response.final_output)

Your name is Hardeep!


#### Another Approach: use OpenAI Agents SDK built in SQLLite session

In [ ]:
# setting up a memory session
session = SQLiteSession("123")

In [18]:
response = await Runner.run(
    starting_agent = agent, 
    input = "Hi there, my name is Hardeep.",
    session = session
)
print(response.final_output)

Hello, Hardeep! Nice to meet you. How can I assist you today?


In [19]:
response = await Runner.run(
    starting_agent = agent, 
    input = "What's my name..?",
    session = session
)
print(response.final_output)

Your name is Hardeep.
